KUL H02A5a Computer Vision: Group Assignment 2
---------------------------------------------------------------

<span style="color:red">**TODO**: Add your names below.</span>

<span style="color:red">**TODO**: Change the notebook name by replacing the X with your actual group number.</span>

Student names: <span style="color:red">name1, name2, ...</span>.

In this group assignment your team will delve into some deep learning applications for computer vision. The assignment will be delivered in the same groups from *Group assignment 1* and you start from this template notebook. You can make use of the *Group assignment 2* forum/discussion board on Toledo if you have any questions.

The notebook you submit for grading is the last notebook pinned as default and submitted to the competition prior to the deadline.

Good luck and have fun!

---------------------------------------------------------------
NOTES:
* This notebook is just a template. Please keep the five main sections, but feel free to adjust further in any way you please!
* Clearly indicate the improvements that you make! You can for instance use subsections like: *3.1. Improvement: applying loss function f instead of g*.


# Overview
This assignment consists of *three main parts* for which we expect you to provide code and extensive documentation in the notebook, with a final discussion:
* Image classification (Sect. 1)
* Semantic segmentation (Sect. 2)
* Adversarial attacks (Sect. 3)
* Discussion (Sect. 4)

## Deep learning resources
If you did not yet explore this in *Group assignment 1 (Sect. 2)*, we recommend using the Pytorch or TensorFlow (and/or Keras) library for building deep learning models.

## Environment setup
Kaggle's bundled PyTorch is sometimes ahead of the GPU it assigns and drops support for older compute capabilities (e.g. the P100 has `sm_60`, which the newest torch wheels no longer ship). Installing a CUDA 12.1 wheel covers Pascal/Volta/Ampere/Hopper GPUs.

**After running this cell, restart the kernel** (Run menu → Restart kernel) so the new torch is picked up.


In [ ]:
!pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/cuda/cu121

## Imports & globals


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
import numpy as np
import pandas as pd
# Uncomment the one you'll use
#import tensorflow as tf
import torch
import torch.nn.functional as F
from tqdm import tqdm
from matplotlib import pyplot as plt


## PASCAL VOC 2009
For this project you will be using the [PASCAL VOC 2009](http://host.robots.ox.ac.uk/pascal/VOC/voc2009/index.html) dataset. This dataset consists of colour images of various scenes with different object classes (e.g. animal: *bird, cat, ...*; vehicle: *aeroplane, bicycle, ...*), totalling 20 classes.

### Loading the training set
Each row of `train_df` holds the image, the per-pixel ground-truth segmentation mask, and 20 binary "is class X present" classification labels. Images and masks are kept as native-resolution NumPy arrays — resizing happens later inside the dataset wrapper.


In [ ]:
# Loading the training data
train_df = pd.read_csv('/kaggle/input/competitions/kul-computer-vision-ga-2-2026/train/train_set.csv', index_col="Id")
labels = train_df.columns
train_df["img"] = [np.load('/kaggle/input/competitions/kul-computer-vision-ga-2-2026/train/img/train_{}.npy'.format(idx)) for idx, _ in train_df.iterrows()]
train_df["seg"] = [np.load('/kaggle/input/competitions/kul-computer-vision-ga-2-2026/train/seg/train_{}.npy'.format(idx)) for idx, _ in train_df.iterrows()]
print("The training set contains {} examples.".format(len(train_df)))

# Show some examples
fig, axs = plt.subplots(2, 20, figsize=(10 * 20, 10 * 2))
for i, label in enumerate(labels):
    df = train_df.loc[train_df[label] == 1]
    axs[0, i].imshow(df.iloc[0]["img"], vmin=0, vmax=255)
    axs[0, i].set_title("\n".join(label for label in labels if df.iloc[0][label] == 1), fontsize=40)
    axs[0, i].axis("off")
    axs[1, i].imshow(df.iloc[0]["seg"], vmin=0, vmax=20)  # with the absolute color scale it will be clear that the arrays in the "seg" column are label maps (labels in [0, 20])
    axs[1, i].axis("off")
    
plt.show()

# The training dataframe contains for each image 20 columns with the ground truth classification labels and 20 column with the ground truth segmentation maps for each class
train_df.head(1)


### Loading the test set
The test set has no ground truth. We initialise `test_df["seg"]` with placeholder values; the segmentation pipeline will overwrite them with the model's predictions.


In [ ]:
# Loading the test data
test_df = pd.read_csv('/kaggle/input/competitions/kul-computer-vision-ga-2-2026/test/test_set.csv', index_col="Id")
test_df["img"] = [np.load('/kaggle/input/competitions/kul-computer-vision-ga-2-2026/test/img/test_{}.npy'.format(idx)) for idx, _ in test_df.iterrows()]
test_df["seg"] = [-1 * np.ones(img.shape[:2], dtype=np.int8) for img in test_df["img"]]
print("The test set contains {} examples.".format(len(test_df)))
test_df.head(1)


## Kaggle submission helpers
Kaggle expects a single `submission.csv` with two rows per test image: one for classification (a 20-bit binary vector) and one for segmentation (a stack of 20 binary class masks). Both are run-length-encoded into one column. `generate_submission(df)` builds this file from a filled `test_df`.


In [ ]:
def _rle_encode(img):
    """RLE encode a binary image array."""
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def generate_submission(df):
    """Convert the filled test dataframe into a Kaggle-format submission.csv."""
    df_dict = {"Id": [], "Predicted": []}
    for idx, _ in df.iterrows():
        df_dict["Id"].append(f"{idx}_classification")
        df_dict["Predicted"].append(_rle_encode(np.array(df.loc[idx, labels])))
        df_dict["Id"].append(f"{idx}_segmentation")
        df_dict["Predicted"].append(_rle_encode(np.array([df.loc[idx, "seg"] == j + 1 for j in range(len(labels))])))
    submission_df = pd.DataFrame(data=df_dict, dtype=str).set_index("Id")
    submission_df.to_csv("submission.csv")
    return submission_df


# 1. Image classification
*(Implemented by another team member.)*

For now, segmentation predictions will be used to derive a baseline for the classification rows of the submission (see Section 2).


# 2. Semantic segmentation

The pipeline supports three architectures behind a single training loop:

- **`UNetScratch`** — classic U-Net trained from scratch (no pretrained weights). Baseline for what end-to-end training can reach without transfer learning.
- **`UNetResNet`** — U-Net with a configurable pretrained ResNet-18..152 encoder. ImageNet pretraining gives the encoder strong low-level features for free.
- **`DeepLabV3+`** — atrous-convolution architecture with a Spatial Pyramid Pooling module, the standard modern baseline on PASCAL VOC.

Loss can be focal or focal+dice combo. ImageNet pretraining is allowed by the assignment because ImageNet contains no PASCAL VOC images.

Layout of section 2:
- 2.0 Design choices
- 2.1 Model definitions
- 2.2 Loss functions
- 2.3 Training infrastructure (`Config`, dataset wrapper, training/prediction loops)
- 2.4 Configuration cell — the only cell to edit between experiments
- 2.5 Train + plot loss/Dice curves
- 2.6 Predict on test + visualise
- 2.7 Derive classification predictions from segmentation
- Submission


## 2.0 Design choices

### How does the task differ from classification?

Classification produces one label vector per image; segmentation produces one label *per pixel*. Two practical consequences:

- The output of a segmentation network has the same spatial dimensions as the input, so the architecture must preserve spatial information rather than collapse it to a vector. We use encoder–decoder architectures (U-Net, DeepLabV3+) where the encoder reduces resolution to extract semantic features and the decoder restores the input resolution.
- The loss is computed per pixel, and the dataset is heavily class-imbalanced at the pixel level (background dominates, and some classes like `sheep` and `tv/monitor` are rare). The loss function has to deal with this imbalance directly, pure cross-entropy works but is dominated by easy background pixels.

### Architecture choices

We compared three architectures, all expressed through a shared `Config` dataclass and training loop so they are directly comparable:

- **`UNetScratch`** — a four-level U-Net trained from scratch. Each block is `(conv → BN → ReLU) × 2`, with max-pool downsampling and transposed-conv upsampling. BatchNorm is added on top of the original 2015 architecture to stabilise training on our small (~640-image) training set. Used as a baseline for what end-to-end training reaches without transfer learning.
- **`UNetResNet`** — the same U-Net topology with the encoder replaced by a pretrained ResNet (any of resnet18/34/50/101/152). The decoder is freshly initialised. ImageNet pretraining is allowed by the assignment because ImageNet contains no PASCAL VOC images. This is the strongest cheap upgrade: ~21 M of the model's parameters start with already-meaningful features (edges, textures, parts) instead of random weights.
- **`DeepLabV3+`** — a different design philosophy, specifically tuned for natural-image segmentation. The deeper encoder stages use **dilated (atrous) convolutions** to keep spatial resolution while expanding the receptive field, an **ASPP** module captures multi-scale context with parallel dilated convolutions (rates 1/6/12/18 plus a global-pooling branch), and a lightweight decoder fuses ASPP output with low-level encoder features.

### Loss function

Our default is a **focal loss** (γ = 2.0) over the 21 classes, optionally combined with a **soft Dice loss** (`ComboLoss`).

- **Focal loss**, `FL = (1 − p_t)^γ · CE`, down-weights well-classified pixels via the `(1 − p_t)^γ` term. With γ = 2.0 and PASCAL VOC's heavy background dominance, this lets the model focus gradient on hard, rare-class pixels rather than getting drowned in the easy majority.
- **Soft Dice loss** is a differentiable surrogate for the metric Kaggle scores us on. It is robust to class imbalance because every class contributes equally to the loss regardless of pixel count. We exclude background from the Dice term to match the Kaggle metric, which scores 20 foreground masks per image.
- **`ComboLoss`** is a weighted sum of the two. The intuition: focal gives stable per-pixel gradients (especially early in training, when the per-class Dice is noisy), and Dice keeps the optimisation directly aligned with the Kaggle metric.

In practice we found that **focal alone, with 30 epochs, was a stronger baseline than combo loss at 60 epochs** for our setup. We discuss why in section 4.

### Learning parameters

- **Optimiser**: AdamW, weight decay 1e-4. AdamW (decoupled weight decay) is more stable than vanilla Adam at non-trivial learning rates.
- **Learning rate**: separate parameter groups for the pretrained encoder (1e-4) and the freshly-initialised decoder (1e-3). The encoder needs gentler updates because its weights are already good; the decoder needs faster updates because it starts from random initialisation.
- **Validation split**: 15 % of training images held out, used to track val Dice and save the best checkpoint. The remaining 85 % is used for training.
- **Augmentation**: horizontal flip, mild brightness/contrast jitter, small affine (±5 % translate, ±10 % scale, ±15° rotation). Image and mask are jointly transformed via `albumentations`.
- **Inference**: the best-by-val-Dice checkpoint is loaded, predictions are produced at original test-image resolution. Logits (not labels) are upsampled to native resolution before argmax, which keeps boundaries sharp.

### Is the model learning as it should?

The training/validation curves in section 2.5 confirm normal learning behaviour: training loss decreases monotonically, validation loss decreases for the first ~20 epochs and plateaus around 25–30. Validation Dice rises in the same pattern. The plot lets us spot two failure modes when they occur: a widening train/val gap signals overfitting, and a flat loss/Dice curve signals underfitting or a learning-rate problem.

Section 2.6 visualises predictions on six test images, which we use as a qualitative sanity check that the model is producing coherent class regions rather than noise.

## 2.1 Model definitions


### `UNetScratch`
Classic U-Net with four down/up levels (channels 64 → 128 → 256 → 512 → 1024). Each block is `(conv → BN → ReLU) × 2`; downsampling is max-pool, upsampling is transposed conv. BatchNorm is added on top of the original 2015 architecture to stabilise training when learning from scratch on a small dataset. He init for the conv layers (standard for ReLU networks).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------------------------------------------------------- #
# Building blocks
# --------------------------------------------------------------------------- #
class DoubleConv(nn.Module):
    """(conv -> BN -> ReLU) x 2.  BatchNorm is added on top of the original
    U-Net to stabilise training when learning from scratch on a small set."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    """Downsample (max-pool /2) followed by a DoubleConv."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.op = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch),
        )

    def forward(self, x):
        return self.op(x)

class Up(nn.Module):
    """Upsample (transposed conv x2) -> concat with the skip from the encoder
    -> DoubleConv.  We use a transposed conv here which
    works slightly better than bilinear upsampling for segmentation in
    practice, at the cost of a few extra parameters."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # Pad if shapes mismatch by 1 pixel (can happen with odd input sizes)
        if x.shape[-2:] != skip.shape[-2:]:
            dy = skip.size(-2) - x.size(-2)
            dx = skip.size(-1) - x.size(-1)
            x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

# --------------------------------------------------------------------------- #
# The full model
# --------------------------------------------------------------------------- #
class UNetScratch(nn.Module):
    """U-Net trained from scratch.

    Parameters
    ----------
    in_channels : int   - 3 for RGB
    num_classes : int   - 21 for PASCAL VOC (background + 20 classes)
    base_ch     : int   - channel count of the first encoder stage; doubles
                          at every level. 64 follows the original paper.
    """

    def __init__(self, in_channels: int = 3, num_classes: int = 21,
                 base_ch: int = 64):
        super().__init__()
        c1, c2, c3, c4, c5 = (base_ch * m for m in (1, 2, 4, 8, 16))

        # Encoder
        self.enc1       = DoubleConv(in_channels, c1)
        self.enc2       = Down(c1, c2)
        self.enc3       = Down(c2, c3)
        self.enc4       = Down(c3, c4)
        self.bottleneck = Down(c4, c5)

        # Decoder
        self.dec4 = Up(c5, skip_ch=c4, out_ch=c4)
        self.dec3 = Up(c4, skip_ch=c3, out_ch=c3)
        self.dec2 = Up(c3, skip_ch=c2, out_ch=c2)
        self.dec1 = Up(c2, skip_ch=c1, out_ch=c1)

        # 1x1 conv head: per-pixel class logits
        self.head = nn.Conv2d(c1, num_classes, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        # He init - good default for ReLU networks
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        s1 = self.enc1(x)            # full res,        c1
        s2 = self.enc2(s1)           # /2,              c2
        s3 = self.enc3(s2)           # /4,              c3
        s4 = self.enc4(s3)           # /8,              c4
        b  = self.bottleneck(s4)     # /16,             c5

        x = self.dec4(b,  s4)        # /8
        x = self.dec3(x,  s3)        # /4
        x = self.dec2(x,  s2)        # /2
        x = self.dec1(x,  s1)        # full res
        return self.head(x)          # (B, num_classes, H, W)

# --------------------------------------------------------------------------- #
# Multi-class focal loss
# --------------------------------------------------------------------------- #
class FocalLoss(nn.Module):
    """Multi-class focal loss for semantic segmentation.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    `gamma` down-weights well-classified pixels - useful when most pixels are
    easy background. `alpha` is an optional per-class weight (tensor of shape
    [num_classes]) that further re-balances rare classes.

    Inputs
    ------
    logits  : (B, C, H, W) raw model outputs
    targets : (B, H, W)    integer class labels in [0, C-1] (use ignore_index
                           to skip pixels, e.g. unlabeled regions if any).
    """

    def __init__(self, gamma: float = 2.0, alpha=None,
                 ignore_index: int = -100, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction
        if alpha is not None and not isinstance(alpha, torch.Tensor):
            alpha = torch.tensor(alpha, dtype=torch.float32)
        self.register_buffer("alpha", alpha) if alpha is not None else None
        self.alpha = alpha  # may be None

    def forward(self, logits, targets):
        # log p_t for every pixel/class: standard CE with reduction='none'
        ce = F.cross_entropy(logits, targets,
                             weight=self.alpha.to(logits.device)
                                    if self.alpha is not None else None,
                             ignore_index=self.ignore_index,
                             reduction="none")            # (B, H, W)
        pt = torch.exp(-ce)                               # = p_t
        loss = (1 - pt) ** self.gamma * ce

        if self.reduction == "mean":
            # mean over non-ignored pixels
            valid = (targets != self.ignore_index)
            return loss[valid].mean() if valid.any() else loss.sum() * 0.0
        if self.reduction == "sum":
            return loss.sum()
        return loss



### `UNetResNet`
U-Net where the encoder is a pretrained ResNet from torchvision (any of resnet18/34/50/101/152). Encoder channel counts are detected automatically from a dummy forward pass — useful because resnet18/34 use BasicBlock (channels [64, 64, 128, 256, 512]) while resnet50/101/152 use Bottleneck (channels [64, 256, 512, 1024, 2048]).

The decoder is built fresh (random init) and bilinear-upsamples through four levels, concatenating the matching encoder skip at each one. Final 1×1 conv maps to `num_classes` logits.

`encoder_parameters()` and `decoder_parameters()` are exposed so the optimiser can use a smaller learning rate for the pretrained encoder than for the random decoder.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# --------------------------------------------------------------------------- #
# Backbone registry
# --------------------------------------------------------------------------- #
_BACKBONES = {
    "resnet18":  (models.resnet18,  models.ResNet18_Weights.IMAGENET1K_V1),
    "resnet34":  (models.resnet34,  models.ResNet34_Weights.IMAGENET1K_V1),
    "resnet50":  (models.resnet50,  models.ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V2),
    "resnet152": (models.resnet152, models.ResNet152_Weights.IMAGENET1K_V2),
}

def _build_backbone(name: str, pretrained: bool):
    if name not in _BACKBONES:
        raise ValueError(f"Unknown backbone {name!r}. "
                         f"Pick one of: {list(_BACKBONES)}")
    ctor, weights = _BACKBONES[name]
    return ctor(weights=weights if pretrained else None)

def _detect_encoder_channels(backbone, in_channels: int = 3):
    """Run a dummy tensor through the encoder stages once to read off the
    output channel count of each stage.  Avoids hardcoding numbers per
    architecture."""
    backbone.eval()
    with torch.no_grad():
        x  = torch.zeros(1, in_channels, 64, 64)
        s1 = backbone.relu(backbone.bn1(backbone.conv1(x)))   # /2
        s2 = backbone.layer1(backbone.maxpool(s1))            # /4
        s3 = backbone.layer2(s2)                              # /8
        s4 = backbone.layer3(s3)                              # /16
        b  = backbone.layer4(s4)                              # /32
    return [t.shape[1] for t in (s1, s2, s3, s4, b)]

# --------------------------------------------------------------------------- #
# Decoder building blocks (random init)
# --------------------------------------------------------------------------- #
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    """Bilinear upsample x2 -> concat with the encoder skip -> DoubleConv."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear",
                                align_corners=False)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            dy = skip.size(-2) - x.size(-2)
            dx = skip.size(-1) - x.size(-1)
            x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

# --------------------------------------------------------------------------- #
# The full model
# --------------------------------------------------------------------------- #
class UNetResNet(nn.Module):
    """U-Net with a configurable pretrained ResNet encoder.

    Parameters
    ----------
    backbone : str   - "resnet18" | "resnet34" | "resnet50" | "resnet101" | "resnet152"
    num_classes : int   - 21 for PASCAL VOC (background + 20 classes)
    pretrained  : bool  - load ImageNet weights for the encoder (default True)
    decoder_channels : tuple[int]
        Channel count at each decoder stage (deep -> shallow). Default
        (256, 128, 64, 32) works well across backbones; you can shrink it for
        speed or grow it for accuracy.
    freeze_encoder : bool - freeze encoder parameters (useful for the first
                            few epochs while the random decoder warms up)
    """

    def __init__(self,
                 backbone: str = "resnet50",
                 num_classes: int = 21,
                 pretrained: bool = True,
                 decoder_channels=(256, 128, 64, 32),
                 freeze_encoder: bool = False):
        super().__init__()
        self.backbone_name = backbone

        bb = _build_backbone(backbone, pretrained)
        enc_ch = _detect_encoder_channels(bb)
        # enc_ch = [s1_ch, s2_ch, s3_ch, s4_ch, bottleneck_ch]
        # e.g. resnet34:  [64, 64,  128, 256,  512]
        # e.g. resnet50:  [64, 256, 512, 1024, 2048]
        self.encoder_channels = enc_ch

        # ----- Encoder: split the ResNet into stages we keep ----- #
        self.enc1       = nn.Sequential(bb.conv1, bb.bn1, bb.relu)  # /2
        self.enc2       = nn.Sequential(bb.maxpool, bb.layer1)      # /4
        self.enc3       = bb.layer2                                  # /8
        self.enc4       = bb.layer3                                  # /16
        self.bottleneck = bb.layer4                                  # /32

        if freeze_encoder:
            for module in (self.enc1, self.enc2, self.enc3,
                           self.enc4, self.bottleneck):
                for p in module.parameters():
                    p.requires_grad = False

        # ----- Decoder (random init) ----- #
        d1, d2, d3, d4 = decoder_channels
        s1, s2, s3, s4, b = enc_ch

        self.dec4 = UpBlock(in_ch=b,  skip_ch=s4, out_ch=d1)   # /32 -> /16
        self.dec3 = UpBlock(in_ch=d1, skip_ch=s3, out_ch=d2)   # /16 -> /8
        self.dec2 = UpBlock(in_ch=d2, skip_ch=s2, out_ch=d3)   # /8  -> /4
        self.dec1 = UpBlock(in_ch=d3, skip_ch=s1, out_ch=d4)   # /4  -> /2

        # The encoder's first stage is at /2 (not /1), so one more upsample
        # is needed to recover full input resolution.
        self.final_up   = nn.Upsample(scale_factor=2, mode="bilinear",
                                      align_corners=False)
        self.final_conv = nn.Sequential(
            nn.Conv2d(d4, d4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(d4),
            nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(d4, num_classes, kernel_size=1)

        self._init_decoder_weights()

    def _init_decoder_weights(self):
        """Only init the decoder. Encoder weights came from ImageNet."""
        for module in (self.dec4, self.dec3, self.dec2, self.dec1,
                       self.final_conv, self.head):
            for m in module.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                            nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    # ----- Convenience helpers for parameter groups ----- #
    def encoder_parameters(self):
        out = []
        for module in (self.enc1, self.enc2, self.enc3,
                       self.enc4, self.bottleneck):
            out.extend(module.parameters())
        return out

    def decoder_parameters(self):
        enc_ids = {id(p) for p in self.encoder_parameters()}
        return [p for p in self.parameters() if id(p) not in enc_ids]

    def forward(self, x):
        s1 = self.enc1(x)            # /2
        s2 = self.enc2(s1)           # /4
        s3 = self.enc3(s2)           # /8
        s4 = self.enc4(s3)           # /16
        b  = self.bottleneck(s4)     # /32

        x = self.dec4(b,  s4)        # /16
        x = self.dec3(x,  s3)        # /8
        x = self.dec2(x,  s2)        # /4
        x = self.dec1(x,  s1)        # /2
        x = self.final_up(x)         # /1
        x = self.final_conv(x)
        return self.head(x)          # (B, num_classes, H, W)


class UNetResNet34(UNetResNet):
    """Convenience alias for the original resnet34 variant."""
    def __init__(self, num_classes=21, pretrained=True, freeze_encoder=False):
        super().__init__(backbone="resnet34", num_classes=num_classes,
                         pretrained=pretrained, freeze_encoder=freeze_encoder)

# --------------------------------------------------------------------------- #
# Multi-class focal loss
# --------------------------------------------------------------------------- #
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha=None,
                 ignore_index: int = -100, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction
        if alpha is not None and not isinstance(alpha, torch.Tensor):
            alpha = torch.tensor(alpha, dtype=torch.float32)
        self.alpha = alpha

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets,
                             weight=self.alpha.to(logits.device)
                                    if self.alpha is not None else None,
                             ignore_index=self.ignore_index,
                             reduction="none")
        pt   = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean":
            valid = (targets != self.ignore_index)
            return loss[valid].mean() if valid.any() else loss.sum() * 0.0
        if self.reduction == "sum":
            return loss.sum()
        return loss

# --------------------------------------------------------------------------- #
# ImageNet normalisation constants (used by the data pipeline)
# --------------------------------------------------------------------------- #
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# --------------------------------------------------------------------------- #


### `DeepLabV3+`
Encoder is a ResNet (50/101/152) modified to use dilated convolutions in the deeper stages, so its output stride is 16 (or 8) instead of the default 32 — the bottleneck retains more spatial detail. After the encoder, an Atrous Spatial Pyramid Pooling (ASPP) module applies four parallel dilated 3×3 convs (rates 6/12/18) plus a global-pooling branch, capturing multi-scale context in one layer.

The decoder is light: upsample the ASPP output to /4, concatenate the encoder's low-level /4 features (projected to 48 channels), apply two 3×3 convs, and a 1×1 head. Final logits are bilinearly upsampled to the input resolution.

This architecture was specifically designed for natural-image segmentation with widely varying object scales.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# --------------------------------------------------------------------------- #
# Backbone with dilated convolutions (output stride 16)
# --------------------------------------------------------------------------- #
_BACKBONES = {
    "resnet50":  (models.resnet50,  models.ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V2),
    "resnet152": (models.resnet152, models.ResNet152_Weights.IMAGENET1K_V2),
}

def _replace_stride_with_dilation(module: nn.Module,
                                  dilation: int):
    """Walk through a ResNet layer (a Sequential of Bottleneck blocks) and:
        - set stride=1 wherever it was 2 (so we don't downsample)
        - set conv2.dilation = dilation (so the receptive field still grows)

    This is the standard trick to convert a classification ResNet into a
    feature extractor at a specified output stride.
    """
    for m in module.modules():
        if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
            m.stride = (1, 1)
        # Bottleneck.conv2 is the 3x3 conv we want to dilate
        if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3):
            m.dilation = (dilation, dilation)
            m.padding  = (dilation, dilation)

def _build_dilated_backbone(name: str, pretrained: bool,
                            output_stride: int = 16):
    """Build a ResNet backbone modified to produce features at the requested
    output stride (16 or 8).  Returns the backbone and the channel count of
    its output and of the low-level feature map (conv1 output, /4)."""

    if name not in _BACKBONES:
        raise ValueError(f"DeepLabV3+ supports {list(_BACKBONES)}; "
                         f"got {name!r}. (resnet18/34 use BasicBlock and "
                         f"don't have the bottleneck structure DeepLab+ "
                         f"is normally combined with.)")
    if output_stride not in (8, 16):
        raise ValueError("output_stride must be 8 or 16")

    ctor, weights = _BACKBONES[name]
    bb = ctor(weights=weights if pretrained else None)

    if output_stride == 16:
        # /32 -> /16: only dilate layer4
        _replace_stride_with_dilation(bb.layer4, dilation=2)
    elif output_stride == 8:
        # /32 -> /8: dilate both layer3 and layer4
        _replace_stride_with_dilation(bb.layer3, dilation=2)
        _replace_stride_with_dilation(bb.layer4, dilation=4)

    # Bottleneck-based ResNets produce 2048 ch at the deepest stage and
    # 256 ch at the /4 level (output of layer1).
    high_level_ch = 2048
    low_level_ch  = 256
    return bb, high_level_ch, low_level_ch

# --------------------------------------------------------------------------- #
# Atrous Spatial Pyramid Pooling
# --------------------------------------------------------------------------- #
class _ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=1, padding=0, dilation=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size,
                      padding=padding, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling.

    Five parallel branches:
      - 1x1 conv (no dilation)
      - 3x3 conv, dilation = atrous_rates[0]
      - 3x3 conv, dilation = atrous_rates[1]
      - 3x3 conv, dilation = atrous_rates[2]
      - global average pool -> 1x1 conv -> upsample
    All branches output `out_ch` channels and are concatenated, then a 1x1
    conv reduces the result back to `out_ch`.
    """

    def __init__(self, in_ch: int, out_ch: int = 256,
                 atrous_rates=(6, 12, 18)):
        super().__init__()
        r1, r2, r3 = atrous_rates
        self.b0 = _ConvBNReLU(in_ch, out_ch, kernel_size=1)
        self.b1 = _ConvBNReLU(in_ch, out_ch, kernel_size=3,
                              padding=r1, dilation=r1)
        self.b2 = _ConvBNReLU(in_ch, out_ch, kernel_size=3,
                              padding=r2, dilation=r2)
        self.b3 = _ConvBNReLU(in_ch, out_ch, kernel_size=3,
                              padding=r3, dilation=r3)
        self.b4 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            _ConvBNReLU(in_ch, out_ch, kernel_size=1),
        )
        self.project = nn.Sequential(
            _ConvBNReLU(out_ch * 5, out_ch, kernel_size=1),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        f0 = self.b0(x)
        f1 = self.b1(x)
        f2 = self.b2(x)
        f3 = self.b3(x)
        # Image-level features: pool to 1x1, then upsample back to (h, w)
        f4 = self.b4(x)
        f4 = F.interpolate(f4, size=(h, w), mode="bilinear",
                           align_corners=False)
        return self.project(torch.cat([f0, f1, f2, f3, f4], dim=1))

# --------------------------------------------------------------------------- #
# DeepLabV3+ model
# --------------------------------------------------------------------------- #
class DeepLabV3Plus(nn.Module):
    """DeepLabV3+ with a pretrained ResNet-50/101/152 encoder.

    Parameters
    ----------
    backbone        : "resnet50" | "resnet101" | "resnet152"
    num_classes     : 21 for PASCAL VOC
    pretrained      : load ImageNet weights for the encoder
    output_stride   : 16 (faster, recommended) or 8 (slower, slightly better)
    aspp_channels   : channel count inside ASPP and decoder (default 256)
    decoder_low_level_channels : channels for the projected low-level feature
                                 (default 48, as in the paper)
    atrous_rates    : dilation rates for ASPP. The default (6, 12, 18) is
                      tuned for output_stride=16.  For output_stride=8 the
                      paper uses (12, 24, 36).
    """

    def __init__(self,
                 backbone: str = "resnet50",
                 num_classes: int = 21,
                 pretrained: bool = True,
                 output_stride: int = 16,
                 aspp_channels: int = 256,
                 decoder_low_level_channels: int = 48,
                 atrous_rates=None):
        super().__init__()
        self.backbone_name = backbone
        self.output_stride = output_stride

        bb, high_ch, low_ch = _build_dilated_backbone(
            backbone, pretrained, output_stride)

        if atrous_rates is None:
            atrous_rates = (6, 12, 18) if output_stride == 16 else (12, 24, 36)

        # Encoder stages.  We grab two outputs:
        #   low_level : after layer1 (output stride 4, low_ch channels)
        #   high_level: after layer4 (output stride 16 or 8, high_ch channels)
        self.stem   = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3
        self.layer4 = bb.layer4

        # ASPP at the deepest stage
        self.aspp = ASPP(in_ch=high_ch, out_ch=aspp_channels,
                         atrous_rates=atrous_rates)

        # Project the low-level features (1x1 conv to a small channel count)
        self.low_level_proj = _ConvBNReLU(low_ch, decoder_low_level_channels,
                                          kernel_size=1)

        # Decoder: concat (upsampled ASPP) + low_level_proj, then 3x3 convs
        decoder_in = aspp_channels + decoder_low_level_channels
        self.decoder = nn.Sequential(
            _ConvBNReLU(decoder_in,    aspp_channels, kernel_size=3, padding=1),
            _ConvBNReLU(aspp_channels, aspp_channels, kernel_size=3, padding=1),
            nn.Dropout(0.1),
        )
        self.head = nn.Conv2d(aspp_channels, num_classes, kernel_size=1)

        self._init_decoder_weights()

    # ----- only init the new modules; encoder came from ImageNet ----- #
    def _init_decoder_weights(self):
        for module in (self.aspp, self.low_level_proj,
                       self.decoder, self.head):
            for m in module.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                            nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    # ----- helpers for the optimizer's parameter groups ----- #
    def encoder_parameters(self):
        out = []
        for module in (self.stem, self.layer1, self.layer2,
                       self.layer3, self.layer4):
            out.extend(module.parameters())
        return out

    def decoder_parameters(self):
        enc_ids = {id(p) for p in self.encoder_parameters()}
        return [p for p in self.parameters() if id(p) not in enc_ids]

    def forward(self, x):
        H, W = x.shape[-2:]
        x = self.stem(x)            # /4
        low = self.layer1(x)        # /4   (low-level features)
        x = self.layer2(low)        # /8
        x = self.layer3(x)          # /16  (or /8 if output_stride=8)
        x = self.layer4(x)          # /16  (or /8)

        # ASPP on the deepest features
        x = self.aspp(x)

        # Upsample ASPP output to the low-level resolution (/4)
        x = F.interpolate(x, size=low.shape[-2:],
                          mode="bilinear", align_corners=False)
        low = self.low_level_proj(low)
        x = torch.cat([x, low], dim=1)

        # Decoder + head
        x = self.decoder(x)
        x = self.head(x)

        # Upsample final logits to the input resolution
        x = F.interpolate(x, size=(H, W),
                          mode="bilinear", align_corners=False)
        return x

# --------------------------------------------------------------------------- #
# Sanity check + parameter counts
# --------------------------------------------------------------------------- #


## 2.2 Loss functions

- **`FocalLoss`** (defined inside the `UNetResNet` cell above) — multi-class focal loss. Down-weights well-classified pixels via `(1−pₜ)^γ`; with γ=2.0 and PASCAL VOC's heavy background dominance, this lets the model focus gradient on the hard, rare-class pixels.
- **`DiceLoss`** — multi-class soft Dice computed over softmax probabilities. Differentiable surrogate for the metric Kaggle scores us on. By default we exclude background, since the Kaggle metric also excludes it from segmentation rows.
- **`ComboLoss`** — weighted sum of the two. Gives stable per-pixel gradients from focal loss and metric-aligned optimisation from Dice. Default weighting 50/50.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    """Multi-class soft Dice loss, computed over softmax probabilities.

    For each class c independently:
        dice_c = (2 * sum(p_c * y_c) + eps) / (sum(p_c) + sum(y_c) + eps)
        loss_c = 1 - dice_c
    Then averaged over classes.

    Notes
    -----
    * Uses softmax (not argmax) so the loss is differentiable.
    """

    def __init__(self,
                 num_classes: int = 21,
                 include_background: bool = False,
                 ignore_index: int = -100,
                 eps: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.include_background = include_background
        self.ignore_index = ignore_index
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # logits  : (B, C, H, W)
        # targets : (B, H, W)
        probs = F.softmax(logits, dim=1)                                # (B, C, H, W)

        # Build one-hot target. Mask out ignored pixels so they don't
        # contribute to either numerator or denominator.
        valid_mask = (targets != self.ignore_index)                     # (B, H, W)
        # Replace ignored labels with 0 so one_hot doesn't crash, then
        # mask their contribution out below.
        safe_targets = targets.clone()
        safe_targets[~valid_mask] = 0
        target_oh = F.one_hot(safe_targets, num_classes=self.num_classes)
        target_oh = target_oh.permute(0, 3, 1, 2).float()               # (B, C, H, W)

        # Apply the valid-pixel mask to both predictions and targets
        m = valid_mask.unsqueeze(1).float()                             # (B, 1, H, W)
        probs     = probs     * m
        target_oh = target_oh * m

        # Class range: optionally skip background (class 0)
        c_start = 0 if self.include_background else 1

        # Per-image, per-class Dice.  Sum over spatial dims, then take
        # mean over classes and batch.
        dims = (2, 3)                                                   # H, W
        inter = (probs[:, c_start:] * target_oh[:, c_start:]).sum(dim=dims)
        denom = probs[:, c_start:].sum(dim=dims) + \
                target_oh[:, c_start:].sum(dim=dims)
        dice  = (2 * inter + self.eps) / (denom + self.eps)             # (B, C')
        return 1.0 - dice.mean()

# FocalLoss is already defined in the unet_resnet cell above, re-used here.

class ComboLoss(nn.Module):
    """Weighted sum of focal loss and Dice loss.

    Parameters
    ----------
    focal_weight : float - weight on focal term (default 0.5)
    dice_weight  : float - weight on Dice term  (default 0.5)
    gamma        : float - focal-loss focusing parameter
    num_classes  : int   - 21 for PASCAL VOC
    include_background_in_dice : bool
        Whether the Dice term sums over background. The Kaggle metric for
        this assignment excludes background from segmentation rows, so
        leaving this False usually matches the leaderboard better.
    ignore_index : int   - pixels with this label are excluded from both
                           terms (use 255 if your data has unlabeled
                           boundary pixels; otherwise leave at -100).
    """

    def __init__(self,
                 focal_weight: float = 0.5,
                 dice_weight: float = 0.5,
                 gamma: float = 2.0,
                 num_classes: int = 21,
                 include_background_in_dice: bool = False,
                 ignore_index: int = -100):
        super().__init__()
        self.focal_weight = focal_weight
        self.dice_weight  = dice_weight
        self.focal = FocalLoss(gamma=gamma, ignore_index=ignore_index)
        self.dice  = DiceLoss(num_classes=num_classes,
                              include_background=include_background_in_dice,
                              ignore_index=ignore_index)

    def forward(self, logits, targets):
        return (self.focal_weight * self.focal(logits, targets) +
                self.dice_weight  * self.dice(logits, targets))

# --------------------------------------------------------------------------- #
# Sanity check
# --------------------------------------------------------------------------- #


## 2.3 Training infrastructure

This cell defines:

- **`Config`** — dataclass holding *all* tunable parameters.
- **`VOCDataset`** — wraps the dataframe rows into a PyTorch dataset, applies `albumentations` augmentations (resize, horizontal flip, brightness/contrast jitter, small affine) to image and mask jointly, and applies ImageNet normalisation when using a pretrained encoder.
- **`mean_dice`** — per-class Dice averaged over classes present in the batch. Used for validation only.
- **`train_model`** — train/val split (15% held out), build model + optimiser (with separate LR groups for encoder/decoder when relevant), cosine LR schedule, focal/combo loss, save best-by-val-Dice checkpoint, optional early stopping when val Dice plateaus.
- **`predict_test`** — load best checkpoint, predict on test images at native resolution with horizontal-flip TTA (forward on image and its mirror, average the logits, then argmax). Logits — not labels — are upsampled, which keeps boundaries sharp.
- **`derive_classification_from_seg`** — heuristic that fills the classification columns based on which classes appear in the predicted mask, used as a free baseline for the classification rows of the submission.


In [ ]:
import os, random
from dataclasses import dataclass, field
from typing import List, Optional
from torch.utils.data import Dataset, DataLoader

# Augmentation: albumentations handles image+mask jointly.
try:
    import albumentations as A
    _ = A.Compose([A.Resize(8, 8)])
    HAS_ALBU = True
except Exception:
    HAS_ALBU = False
    print("[warn] albumentations not available; falling back to manual aug.")

@dataclass
class Config:
    # Model
    model_name:    str   = "deeplab"        # "deeplab" | "resnet" | "scratch"
    backbone:      str   = "resnet50"       # "resnet18..152" for "resnet", "resnet50/101/152" for "deeplab"
    num_classes:   int   = 21               # background + 20 VOC classes
    output_stride: int   = 16               # only used when model_name == "deeplab" (8 or 16)

    # Data / training
    img_size:     int   = 256
    batch_size:   int   = 12
    num_epochs:   int   = 30
    val_fraction: float = 0.15
    seed:         int   = 42
    num_workers:  int   = 4

    # Optimisation
    lr_decoder:   float = 1e-3
    lr_encoder:   float = 1e-4
    weight_decay: float = 1e-4

    # Loss: "focal" or "combo" (focal+dice)
    loss_kind:    str   = "combo"
    focal_gamma:  float = 2.0
    focal_weight: float = 0.5
    dice_weight:  float = 0.5

    # Early stopping (set patience = None to disable)
    patience:     Optional[int] = 10        # stop if no val_dice improvement for this many epochs

    # Runtime
    device:       str   = field(
        default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    checkpoint:   str   = "/kaggle/working/best_model.pt"

    def short_name(self):
        arch = self.model_name
        if arch == "deeplab":
            arch = f"deeplab(os{self.output_stride})"
        return (f"{arch}-{self.backbone}-"
                f"img{self.img_size}-bs{self.batch_size}-ep{self.num_epochs}-"
                f"loss_{self.loss_kind}")

def _set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ----- Dataset ----- #
class VOCDataset(Dataset):
    def __init__(self, images, masks=None, img_size=256,
                 augment=False, normalize_imagenet=True):
        self.images = images; self.masks = masks
        self.img_size = img_size; self.augment = augment
        self.normalize_imagenet = normalize_imagenet
        self.tf = None
        if HAS_ALBU:
            try:
                if augment:
                    self.tf = A.Compose([
                        A.Resize(img_size, img_size),
                        A.HorizontalFlip(p=0.5),
                        A.RandomBrightnessContrast(p=0.3),
                        A.Affine(translate_percent=(-0.05, 0.05),
                                 scale=(0.9, 1.1),
                                 rotate=(-15, 15), p=0.5),
                    ])
                else:
                    self.tf = A.Compose([A.Resize(img_size, img_size)])
            except Exception as e:
                print(f"[warn] albumentations failed: {e!r}; using fallback.")
                self.tf = None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        msk = self.masks[idx] if self.masks is not None \
              else np.zeros(img.shape[:2], dtype=np.int64)
        if self.tf is not None:
            out = self.tf(image=img, mask=msk)
            img, msk = out["image"], out["mask"]
        else:
            img = self._resize_image(img, self.img_size)
            msk = self._resize_mask(msk, self.img_size)
            if self.augment and random.random() < 0.5:
                img = img[:, ::-1, :].copy(); msk = msk[:, ::-1].copy()
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        if self.normalize_imagenet:
            mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
            std  = torch.tensor(IMAGENET_STD ).view(3, 1, 1)
            img = (img - mean) / std
        msk = torch.from_numpy(np.asarray(msk)).long()
        return img, msk

    @staticmethod
    def _resize_image(img, size):
        t = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0)
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        return t.squeeze(0).permute(1, 2, 0).numpy().astype(np.uint8)

    @staticmethod
    def _resize_mask(msk, size):
        t = torch.from_numpy(msk).unsqueeze(0).unsqueeze(0).float()
        t = F.interpolate(t, size=(size, size), mode="nearest")
        return t.squeeze(0).squeeze(0).numpy().astype(np.int64)

# ----- Validation Dice ----- #
@torch.no_grad()
def mean_dice(logits, targets, num_classes=21, eps=1e-7):
    preds = logits.argmax(dim=1)
    dices = []
    for c in range(num_classes):
        pred_c = (preds == c); targ_c = (targets == c)
        if targ_c.sum() == 0 and pred_c.sum() == 0:
            continue
        inter = (pred_c & targ_c).sum().float()
        denom = pred_c.sum().float() + targ_c.sum().float()
        dices.append(((2 * inter + eps) / (denom + eps)).item())
    return float(np.mean(dices)) if dices else 0.0

# ----- Model factory ----- #
def _build_model_and_optim(cfg):
    if cfg.model_name == "resnet":
        model = UNetResNet(backbone=cfg.backbone,
                           num_classes=cfg.num_classes, pretrained=True)
        normalize = True
    elif cfg.model_name == "deeplab":
        model = DeepLabV3Plus(backbone=cfg.backbone,
                              num_classes=cfg.num_classes,
                              pretrained=True,
                              output_stride=cfg.output_stride)
        normalize = True
    elif cfg.model_name == "scratch":
        model = UNetScratch(num_classes=cfg.num_classes)
        normalize = False
    else:
        raise ValueError(f"unknown model_name: {cfg.model_name}")

    if cfg.model_name in ("resnet", "deeplab"):
        # Smaller LR for the pretrained encoder, larger for the random decoder
        optim = torch.optim.AdamW([
            {"params": model.encoder_parameters(), "lr": cfg.lr_encoder},
            {"params": model.decoder_parameters(), "lr": cfg.lr_decoder},
        ], weight_decay=cfg.weight_decay)
    else:
        optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr_decoder,
                                  weight_decay=cfg.weight_decay)
    return model.to(cfg.device), optim, normalize

def _build_loss(cfg):
    if cfg.loss_kind == "focal":
        return FocalLoss(gamma=cfg.focal_gamma)
    elif cfg.loss_kind == "combo":
        return ComboLoss(focal_weight=cfg.focal_weight,
                         dice_weight=cfg.dice_weight,
                         gamma=cfg.focal_gamma,
                         num_classes=cfg.num_classes,
                         include_background_in_dice=False)
    else:
        raise ValueError(f"unknown loss_kind: {cfg.loss_kind}")

def train_model(train_df, cfg=None):
    cfg = cfg or Config()
    print(f"=== {cfg.short_name()} on {cfg.device} ===")
    _set_seed(cfg.seed)

    n_total = len(train_df)
    idx = np.arange(n_total); np.random.shuffle(idx)
    n_val = int(cfg.val_fraction * n_total)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    images = list(train_df["img"]); masks = list(train_df["seg"])
    train_imgs  = [images[i] for i in train_idx]
    train_masks = [masks [i] for i in train_idx]
    val_imgs    = [images[i] for i in val_idx]
    val_masks   = [masks [i] for i in val_idx]
    print(f"train: {len(train_imgs)} | val: {len(val_imgs)}")

    model, optim, normalize = _build_model_and_optim(cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model params: {n_params/1e6:.1f}M")

    train_ds = VOCDataset(train_imgs, train_masks, cfg.img_size,
                          augment=True,  normalize_imagenet=normalize)
    val_ds   = VOCDataset(val_imgs,   val_masks,   cfg.img_size,
                          augment=False, normalize_imagenet=normalize)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True,
                              drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=True)

    loss_fn   = _build_loss(cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg.num_epochs)

    best_val_dice = -1.0
    epochs_since_best = 0
    history = {"train_loss": [], "val_loss": [], "val_dice": []}

    for epoch in range(1, cfg.num_epochs + 1):
        # train
        model.train(); running = 0.0
        for imgs, msks in train_loader:
            imgs = imgs.to(cfg.device, non_blocking=True)
            msks = msks.to(cfg.device, non_blocking=True)
            logits = model(imgs); loss = loss_fn(logits, msks)
            optim.zero_grad(); loss.backward(); optim.step()
            running += loss.item() * imgs.size(0)
        scheduler.step()
        train_loss = running / len(train_ds)

        # validate
        model.eval(); val_loss_sum = 0.0; dices = []
        with torch.no_grad():
            for imgs, msks in val_loader:
                imgs = imgs.to(cfg.device, non_blocking=True)
                msks = msks.to(cfg.device, non_blocking=True)
                logits = model(imgs)
                val_loss_sum += loss_fn(logits, msks).item() * imgs.size(0)
                dices.append(mean_dice(logits, msks, cfg.num_classes))
        val_loss = val_loss_sum / len(val_ds)
        val_dice = float(np.mean(dices)) if dices else 0.0

        history["train_loss"].append(train_loss)
        history["val_loss"  ].append(val_loss)
        history["val_dice"  ].append(val_dice)

        marker = ""
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), cfg.checkpoint)
            marker = "  <- best, saved"
            epochs_since_best = 0
        else:
            epochs_since_best += 1
        print(f"epoch {epoch:3d}/{cfg.num_epochs}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_dice={val_dice:.4f}{marker}")

        if cfg.patience is not None and epochs_since_best >= cfg.patience:
            print(f"early stopping at epoch {epoch}: "
                  f"no improvement for {cfg.patience} epochs")
            break

    print(f"\nbest val Dice: {best_val_dice:.4f}  (checkpoint: {cfg.checkpoint})")
    return model, history

@torch.no_grad()
def predict_test(test_df, cfg=None):
    cfg = cfg or Config()
    if cfg.model_name == "resnet":
        model = UNetResNet(backbone=cfg.backbone,
                           num_classes=cfg.num_classes, pretrained=False)
        normalize = True
    elif cfg.model_name == "deeplab":
        model = DeepLabV3Plus(backbone=cfg.backbone,
                              num_classes=cfg.num_classes,
                              pretrained=False,
                              output_stride=cfg.output_stride)
        normalize = True
    else:
        model = UNetScratch(num_classes=cfg.num_classes)
        normalize = False
    model.load_state_dict(torch.load(cfg.checkpoint, map_location=cfg.device))
    model.to(cfg.device).eval()

    if normalize:
        mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(cfg.device)
        std  = torch.tensor(IMAGENET_STD ).view(1, 3, 1, 1).to(cfg.device)

    preds = []
    for img in test_df["img"]:
        H0, W0, _ = img.shape
        x = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        x = F.interpolate(x, size=(cfg.img_size, cfg.img_size),
                          mode="bilinear", align_corners=False).to(cfg.device)
        if normalize:
            x = (x - mean) / std
        # Horizontal-flip TTA
        logits1 = model(x); logits2 = model(torch.flip(x, dims=[-1]))
        logits  = (logits1 + torch.flip(logits2, dims=[-1])) / 2
        # Upsample logits, then argmax
        logits = F.interpolate(logits, size=(H0, W0),
                               mode="bilinear", align_corners=False)
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int8)
        preds.append(pred)

    test_df = test_df.copy()
    test_df["seg"] = preds
    return test_df

def derive_classification_from_seg(test_df, labels, min_pixels=50):
    """Free baseline for the classification rows: a class is "present"
    iff its segmentation mask has >= min_pixels pixels."""
    test_df = test_df.copy()
    for idx, row in test_df.iterrows():
        seg = row["seg"]
        for j, lbl in enumerate(labels):
            test_df.at[idx, lbl] = int((seg == j + 1).sum() >= min_pixels)
    return test_df


## 2.4 Configuration

**This cell enables switching architectures, changing hyperparameters, etc.** Cells below it read `cfg` and use whatever is set here.

Architecture choices:
- `model_name="deeplab"` with `backbone="resnet50"` or `"resnet101"`
- `model_name="resnet"` (any ResNet-18..152) for U-Net + ImageNet pretrained encoder.
- `model_name="scratch"` for the from-scratch U-Net baseline.

Loss: `loss_kind="combo"` (focal+dice) or `"focal"` alone.



In [ ]:
cfg = Config(
    # Model
    model_name    = "deeplab",      # "deeplab" | "resnet" | "scratch"
    backbone      = "resnet101",     # "resnet50" / "resnet101" / "resnet152" for deeplab
    output_stride = 16,             # 16 (faster) or 8 (slightly better, much slower)

    # Data / training
    img_size    = 384,
    batch_size  = 12,
    num_epochs  = 60,
    num_workers = 4,

    # Loss
    loss_kind    = "focal",         # "combo" or "focal"
    focal_weight = 0.5,
    dice_weight  = 0.5,

    # Early stopping
    patience = 30,                  # stop if no val_dice improvement for this many epochs

    # Output
    checkpoint = "/kaggle/working/best_model.pt",
)
print(cfg)


## 2.5 Train

Runs the training loop and plots train/val loss + val Dice curves. Watch the val Dice plot for convergence (plateau = consider stopping early next time) and the train/val loss gap (large gap = overfitting). The best checkpoint by val Dice is saved automatically.


In [ ]:
model, history = train_model(train_df, cfg)

# Plot the curves so you can spot overfitting / convergence
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"],   label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Loss")
plt.subplot(1, 2, 2)
plt.plot(history["val_dice"])
plt.xlabel("epoch"); plt.ylabel("mean Dice (val)")
plt.title(f"Best val Dice: {max(history['val_dice']):.4f}")
plt.tight_layout(); plt.show()


## 2.6 Predict on the test set & visualise

Loads the best checkpoint, runs inference at each test image's original resolution with horizontal-flip TTA, and shows six example predictions next to the input images for sanity-checking. The visualisation uses the `tab20` colormap so each class gets a distinct colour.


In [ ]:
test_df = predict_test(test_df, cfg)

# Show a few example predictions for sanity-checking
fig, axs = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axs[0, i].imshow(test_df.iloc[i]["img"])
    axs[0, i].axis("off"); axs[0, i].set_title(f"test_{test_df.index[i]}")
    axs[1, i].imshow(test_df.iloc[i]["seg"], vmin=0, vmax=20, cmap="tab20")
    axs[1, i].axis("off")
plt.tight_layout(); plt.show()


## 2.7 Derive classification predictions from segmentation

The submission CSV needs both classification and segmentation rows. Until the team's classification model is plugged in, this heuristic gives a reasonable baseline: a class is "present" if its predicted segmentation mask covers at least 50 pixels. If the classification predictions are coming from a separate model, this cell can be replaced with code that overwrites the `labels` columns from that model.


In [ ]:
test_df = derive_classification_from_seg(test_df, labels, min_pixels=50)
test_df.head(3)


## Submit
Writes `/kaggle/working/submission.csv` in the format Kaggle expects (interleaved classification/segmentation rows, RLE-encoded). After running, hit "Submit" on the Kaggle competition page to score it on the leaderboard.


In [ ]:
generate_submission(test_df)


# 3. Adversarial attack on the best-performing segmentation model
**Best-performing model**  

Our final segmentation submission uses Deeplab with a resnet 101 backbone and an image input size of 384, trained for 60 epochs on the 2 250-image training set with a 15 % held-out validation split. Best Kaggle score **0.829**.

We choose to perform untargeted attack on this model.

**Method: Projected Gradient Descent (PGD) method**  
Goal: Reduce the best model's prediction accuracy dramatically by adding noise to the original test image, so the model can no longer predict correct lables for large amounts of pixels in images.

**Result**  
We use the PGD method to generate perturbed segmentation masks for the whole test dataset.
The predicted masks score 18.23% in Kaggle, which is a drop from 82.00%, suggesting that our attack has been very effective.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(device)
std  = torch.tensor(IMAGENET_STD ).view(1, 3, 1, 1).to(device)

def pgd_attack(model, image_vector, mask_vector):
    # Convert numpy to tensor
    original_mask = torch.from_numpy(mask_vector).long().unsqueeze(0).to(device)

    H0, W0, _ = image_vector.shape

    # Image preprocessing so the image can be provided to the model
    x = torch.from_numpy(image_vector).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    x = F.interpolate(x, size=(384, 384),
                    mode="bilinear", align_corners=False)
    x = x.to(device)
    x = (x - mean) / std

    # Set model to eval mode and disable gradients for the model parameters
    model.eval()
    # Use the loss function from the training configuration
    loss_fn = FocalLoss()

    eps = 0.03 # This value can be smaller. The perturbed image is a bit blurred compared with the original one. Since we want to know how strong the attack can be while the perturbed image still lools natural, we choose this parameter.
    alpha = 0.005   # step size
    steps = 60
    prev_loss = None
    x_adv = x.clone().detach()

    for i in range(steps):

        x_adv.requires_grad_(True)

        # forward (TTA included); This follows how the trained model compute logits for each test image. We could also compute logits1 and upsampling it directly to the final mask size, which is how logits were computed in the training stage. Since we mainly attack test images, we choose the former setting, so the loss compute is more accurate.
        logits1 = model(x_adv)
        logits2 = model(torch.flip(x_adv, dims=[-1]))

        logits = (logits1 + torch.flip(logits2, dims=[-1])) / 2

        logits = F.interpolate(logits,
                                size=(H0, W0),
                                mode="bilinear",
                                align_corners=False)

        # compute loss
        loss = loss_fn(logits, original_mask)

        current_loss = loss.item()
        # early stopping setting
        if prev_loss is not None and prev_loss - current_loss > 0:
            print(f"Early stopping at step {i}: loss decreases instead of increasing.")
            break

        prev_loss = current_loss

        # backward
        loss.backward()

        # PGD update: Update the image by moving it in the direction that increases the loss.
        grad = x_adv.grad.sign()# Get the gradient of the loss with respect to the input image. But only take the direction of the gradient, discard magnitude
        x_adv = x_adv + alpha * grad

        # projection (keep within epsilon ball so the image is not too distorted)
        x_adv = torch.max(torch.min(x_adv, x + eps), x - eps)

        x_adv = x_adv.detach()

        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int8)

    return pred, x_adv

In [ ]:
new_test_df = test_df.copy()
new_test_df = new_test_df[:5] # Here we pick the first 5 test images for running on Kaggle
print(len(new_test_df))
perturbed_preds = []
x_adv_list = []

for i in tqdm(range(len(new_test_df))):
    img_vector = new_test_df["img"][i]
    mask_vector = new_test_df["seg"][i]
    pred, x_adv = pgd_attack(model, img_vector, mask_vector)

    perturbed_preds.append(pred)
    x_adv_list.append(x_adv.detach().cpu())

# save the perturbed mask and the processed perturbed image
new_test_df["perturbed_seg"] = perturbed_preds
new_test_df["x_adv_list"] = x_adv_list # It is better to save the tensor file in .pt format or converted tensor to numpy and save into csv. But here we put it together with other data for quick progress.
new_test_df.head(3)

In [ ]:
def derive_classification_from_seg1(test_df, labels, min_pixels=50):
    """Free baseline for the classification rows: a class is "present"
    iff its segmentation mask has >= min_pixels pixels."""
    test_df = test_df.copy()
    for idx, row in test_df.iterrows():
        seg = row["perturbed_seg"]
        for j, lbl in enumerate(labels):
            test_df.at[idx, lbl] = int((seg == j + 1).sum() >= min_pixels)
    return test_df

def generate_submission1(df):
    """Convert the filled test dataframe into a Kaggle-format submission.csv."""
    df_dict = {"Id": [], "Predicted": []}
    for idx, _ in df.iterrows():
        df_dict["Id"].append(f"{idx}_classification")
        df_dict["Predicted"].append(_rle_encode(np.array(df.loc[idx, labels])))
        df_dict["Id"].append(f"{idx}_segmentation")
        df_dict["Predicted"].append(_rle_encode(np.array([df.loc[idx, "perturbed_seg"] == j + 1 for j in range(len(labels))])))
    submission_df = pd.DataFrame(data=df_dict, dtype=str).set_index("Id")
    submission_df.to_csv("/kaggle/working/perturbed_submission.csv")
    return submission_df

# We copy two functions to this part so it can be modified more easily
new_test_df = derive_classification_from_seg1(new_test_df, labels, min_pixels=50)
new_test_df.head(3)
generate_submission1(new_test_df)

In [ ]:
def visualize_attack(test_df, idx):

    raw_image = test_df.iloc[idx]["img"]
    print(f"raw_image shape: {raw_image.shape}")
    
    # Before a test image is given to the model, it undergoes preprocessing so that its format matches the model's requirements. Here, we reconstruct the perturbed image back to its state prior to preprocessing. Then we compare the test image with perturbed image.
    x_adv=test_df.iloc[idx]["x_adv_list"]
    H0, W0, _ = raw_image.shape
    x_adv=x_adv.to(device)
    x_inv = (x_adv * std) + mean
    x_inv = F.interpolate(x_inv, size=(H0, W0), mode="bilinear", align_corners=False)
    x_inv = x_inv.squeeze(0).permute(1, 2, 0)
    x_inv = x_inv * 255.0
    perturbed_image = x_inv.detach().cpu().numpy().clip(0, 255).astype(np.uint8)

    print(f"perturbed_image shape: {perturbed_image.shape}")

    raw_mask  = test_df.iloc[idx]["seg"]
    print(f"raw_mask shape: {raw_mask.shape}")

    perturbed_seg = test_df.iloc[idx]["perturbed_seg"]
    print(f"Perturbed_mask shape: {perturbed_seg.shape}")
    # plot
    fig, axs = plt.subplots(1, 5, figsize=(18, 5))

    converted_raw_image = raw_image.astype(np.float32)
    converted_perturbed_image = perturbed_image.astype(np.float32)
    perturbation = converted_perturbed_image - converted_raw_image
    print("min/max perturbation:", perturbation.min(), perturbation.max())
    perturbation_mag = np.mean(np.abs(perturbation), axis=2)

    # axs[0] — Raw image
    axs[0].imshow(raw_image)
    axs[0].set_title("Raw Image")
    axs[0].axis("off")

    # axs[1] — Perturbed image
    axs[1].imshow(perturbed_image)
    axs[1].set_title("Perturbed Image")
    axs[1].axis("off")

    # axs[2]- Perturbation magnitude
    axs[2].imshow(perturbation_mag, cmap="hot")
    axs[2].set_title("Perturbation Magnitude")
    axs[2].axis("off")
    # Compared with FGSM (Fast Gradient Sign Method), PGD noise is more spatially structured. It tends to concentrate perturbations around edges and semantically important regions since it iteratively refines the gradient direction.

    # axs[3] — Original prediction mask
    axs[3].imshow(raw_mask, vmin=0, vmax=20, cmap="tab20")
    axs[3].set_title("Original Prediction Mask")
    axs[3].axis("off")

    # axs[4] — Perturbed segmentation mask
    axs[4].imshow(perturbed_seg, vmin=0, vmax=20, cmap="tab20")
    axs[4].set_title("Perturbed Segmentation Mask")
    axs[4].axis("off")

    plt.tight_layout()
    plt.show()

    return perturbed_image # return perturbed_image for the next function

def compute_stats(test_df, perturbed_image, idx):
    raw_image     = test_df.iloc[idx]["img"]
    raw_mask      = test_df.iloc[idx]["seg"]
    perturbed_seg = test_df.iloc[idx]["perturbed_seg"]
    print("Unique classes:", np.unique(raw_mask))
    print("Unique classes (perturbed):", np.unique(perturbed_seg))

    # Pixel change rate
    perturbation = perturbed_image.astype(np.float32) - raw_image.astype(np.float32)
    changed_pixels = np.any(np.abs(perturbation) > 0, axis=2)
    pixel_change_rate = changed_pixels.sum() / changed_pixels.size * 100

    # Class change rate
    class_change_rate = (raw_mask != perturbed_seg).sum() / raw_mask.size * 100

    print(f"Pixel change rate: {pixel_change_rate:.2f}%")
    print(f"Class change rate: {class_change_rate:.2f}%")

    return pixel_change_rate, class_change_rate

In [ ]:
perturbed_image= visualize_attack(new_test_df,1)
compute_stats(new_test_df,perturbed_image, 1) # The perturbed image looks very similar to original image but a bit blurred

# 4. Discussion (segmentation)

## 4.1 What we built and how it performed

Our final segmentation submission uses Deeplab with a resnet 101 backbone and an image input size of 384, trained for 60 epochs on the 2 250-image training set with a 15 % held-out validation split. Best Kaggle score **0.829**.

Reaching this required a sequence of incremental design decisions which we discuss below, including some that did not work.

## 4.2 Experiments and results

The headline results across our main experiments:

| Architecture | Backbone | Loss | Epochs | Kaggle Dice |
| --- | --- | --- | --- | --- |
| U-Net (scratch) | — | focal | 30 | *(baseline, much lower)* |
| U-Net | resnet101 | focal | 30 | 0.79 |
| U-Net | resnet152 | focal | 30 | 0.785 |
| U-Net | resnet101 | combo (50/50) | 60 | 0.771 |
| U-Net | resnet101 | combo (30/70) | 60 | 0.775 |
| DeepLabV3+ | resnet50 | focal | 30 | 0.753 |
| DeepLabV3+ | resnet101 | focal | 30 | 0.79 |
| U-Net + random-crop aug + multi-scale TTA | resnet101 | focal | 30 | 0.817 |
| **Best submission** | **[FILL IN]** | **[FILL IN]** | **[FILL IN]** | **0.826** |


**Pretrained encoder vs from-scratch U-Net.** ImageNet pretraining was the single largest source of performance. With only ~640 effective training images after the validation split, training a 31 M-parameter U-Net from scratch never reaches the same feature quality. This matches the lecture material on transfer learning: pretrained features encode general visual concepts (edges, textures, parts) that transfer well across datasets, and re-learning them from scratch is wasteful when only a few thousand labels are available.

**Backbone size: diminishing returns.** Going from resnet34 (71.08) to resnet101 (79.3) gave a clear gain, but resnet152 actually scored *lower* (0.785) on Kaggle despite slightly higher validation Dice (0.577 vs 0.55). We interpret this as the bigger model starting to overfit our small training set: validation Dice tracks something the model can memorise more thoroughly, but real generalisation to the held-out Kaggle test set does not improve. This is consistent with the bias–variance trade-off: with limited data, increasing model capacity past a point reduces variance on training but increases variance in generalisation.

**Combo loss at 60 epochs underperformed focal at 30 epochs.** This was unexpected as the literature suggests Dice-aware losses help on Dice-scored tasks. We diagnosed this as overfitting from the longer training run rather than the loss being intrinsically worse: the cosine learning-rate schedule was tuned for the longer run, weight decay had more updates to amplify, and our small training set has limited capacity to absorb the extra epochs.

**Random-crop augmentation and multi-scale TTA.** Switching from fixed-size resize to random-scale (0.5×–2.0×) + random-crop augmentation, combined with multi-scale flip TTA at inference (0.75×, 1.0×, 1.25× × {flipped, unflipped}, average logits), nudged us from ~0.79 to ~0.817. Smaller than expected — random-crop's value typically scales with how aggressive the augmentation is, and our scale range may have been too aggressive given that PASCAL VOC test images are mostly standard-framed.

**Class-weighted loss (failed experiment).** We computed inverse-square-root pixel-frequency weights and passed them as `alpha` to focal loss. Validation loss dropped substantially  but validation Dice also dropped, from ~0.55 to ~0.32. Inspecting predictions revealed the model was over-predicting rare classes everywhere it was uncertain, because rare-class pixels carried weight ~350× larger than background pixels. The weighted loss was happy with this; the Dice metric was not. This is a useful empirical illustration that **the loss function and the evaluation metric do not always pull in the same direction** — a 50 % drop in loss accompanied by a 40 % drop in metric is a clear sign that the loss is no longer aligned with what we actually care about. Removing class weights and keeping the other improvements gave us 0.817; from there, our best submission of 0.826 came from switching to the deeplab architecture and increasing the input image size from 254 to 384.


## 4.3 What we would do with more time

- **Potential problems with architecture.** After printing out the segmentation result from the model, we observe that there are clear issues with boundary precision and structural noise. We could: 
1. Backbone Swap: Consider HRNet (High-Resolution Net). Unlike ResNet, which recovers resolution from low-res features, HRNet maintains high-resolution representations throughout the process, which is vital for the clean lines of objects.
2. Attention Mechanisms: Integrate CBAM (Convolutional Block Attention Module) or use a Transformer-based backbone like SegFormer (MiT-B3 or B5). Transformers excel at global context, helping the model realize that some colored pixels shouldn't exist.
3. Input Resolution: 384 is somewhat small for objects with thin features (like the bottle neck). If GPU memory allows, we can bump this to 512x512 or 640x640. 
4. Augmentation: We could take obects from the training set and "paste" them onto different backgrounds. This forces the model to learn the shape of the object rather than relying on the lighting of a specific scene. 
- **Ensembling.** Train 3–5 models with different random seeds and average their predicted logits at inference time.
- **More careful class-weighted loss.** Our naive inverse-square-root weighting failed because it was too aggressive. Clipping weights to a narrow range (e.g. [0.5, 2.0]) would prevent the rare-class-over-prediction failure mode while still nudging the model toward harder classes.
- **Larger input resolution.** Bumping `img_size` from 384 to even higher values should help with segmenting smaller objects, at the cost of GPU memory and training time.
- **Adversarial attack.** We could train the best model on the perturbed images so it knows how to defend against the attack. 


## 4.6 Limitations and links to lectures


- **Pretraining is doing most of the work.** Most of our model's effective capacity came from ImageNet, not from PASCAL VOC training. Without it, the from-scratch U-Net was much weaker. This connects to the lecture material on transfer learning being almost mandatory at small data scales.

- **Architecture choice mattered less than expected.** With a pretrained encoder, U-Net and DeepLabV3+ converged to similar performance on our 
subset of PASCAL VOC.

- **Real-world deployment.** A model trained on PASCAL VOC photos would not transfer well to other domains: medical imagery, satellite imagery, low-light street scenes, etc. PASCAL VOC photos are well-lit, well-framed, and mostly contain a small number of large foreground objects. For real-world applications, we need larger training datasets and better architecture to improve the model's performance. Also, from the session below we can see that adversarial perturbations will mess with accuracy and we should prepare a model for the potential attack.


## 4.7 Summary

Our best segmentation submission scored **0.829** on Kaggle. The largest single source of performance was using a pretrained ResNet encoder; subsequent tweaks (architecture choice, loss design, augmentation, TTA) each moved the score by a few percent at most. From our experiments it appears that we are more data-limited than architecture-limited at this scale.